# 🎯 Ćwiczenie 7: Spark Structured Streaming — przetwarzanie strumieniowe danych

**Kurs:** Python w Zastosowaniach Big Data — PJATK  
**Data:** 14.06.2026  
**Czas realizacji:** ⏱️ 90 minut  
**Poziom trudności:** ⭐⭐–⭐⭐⭐ (Intermediate-Advanced)  

---

## 📚 Cele kształcenia

Po ukończeniu tego notebooka będziesz potrafił/a:

1. **Tworzyć strumienie danych** za pomocą `readStream` i różnych źródeł (rate, file)
2. **Filtrować i transformować** dane strumieniowe w czasie rzeczywistym
3. **Stosować agregacje** z różnymi output modes (Append, Complete, Update)
4. **Używać okien czasowych** — tumbling windows i sliding windows
5. **Obsługiwać spóźnione dane** dzięki mechanizmowi watermark
6. **Przetwarzać strumienie plikowe** z checkpointingiem
7. **Budować pipeline** alertów w czasie rzeczywistym

## 📋 Wymagania wstępne

- Podstawy PySpark (DataFrames, transformacje, akcje)
- Znajomość Spark SQL (funkcje wbudowane, grupowanie)
- Podstawowe pojęcie o przetwarzaniu danych w czasie rzeczywistym

---

> 💡 **Wskazówka:** Uruchamiaj komórki po kolei od góry do dołu za pomocą `Shift+Enter`.  
> Jeśli utkniesz — kliknij komórkę z podpowiedzią (💡) lub zapytaj prowadzącego!

> ⚠️ **Ważne:** Streaming w Colab wymaga cierpliwości — każdy query potrzebuje kilkunastu sekund na zgromadzenie danych. Używamy `awaitTermination(timeout=30)` żeby automatycznie zatrzymywać strumienie.

In [ ]:
# @title 🔧 Konfiguracja środowiska — uruchom tę komórkę jako pierwszą!

!pip install -q pyspark

# PySpark — sesja i typy
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, TimestampType, DoubleType
)
from pyspark.sql.functions import (
    col, window, expr, current_timestamp, lit,
    count, avg, sum as _sum, max as _max, min as _min,
    when, rand, round as _round, from_json, to_json, struct
)

# Standardowe biblioteki
import time
import os
import shutil
import json
import random
from datetime import datetime, timedelta
from threading import Thread

# Inicjalizacja SparkSession
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('CW07-Structured-Streaming-SmartCity')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.streaming.schemaInference', 'true')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

# Katalogi robocze
CHECKPOINT_DIR = '/tmp/streaming_checkpoints'
DATA_DIR = '/tmp/streaming_data'

# Wyczyść katalogi z poprzednich uruchomień
for d in [CHECKPOINT_DIR, DATA_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

print(f'✅ PySpark {spark.version} gotowy!')
print(f'📌 SparkSession: {spark.sparkContext.appName}')
print(f'📂 Checkpoint dir: {CHECKPOINT_DIR}')
print(f'📂 Data dir: {DATA_DIR}')
print('🚀 Środowisko skonfigurowane — zaczynamy!')

## 📖 Wprowadzenie — Spark Structured Streaming

Wyobraź sobie, że dane to **rzeka** 🏞️ — ciągle płyną i nigdy się nie kończą. Tradycyjne przetwarzanie (batch) to jak zbieranie wody do wiadra i analizowanie jej po fakcie. **Structured Streaming** to jak analizowanie wody *w trakcie przepływu*!

### 🔑 Kluczowe pojęcia

| Pojęcie | Opis | Analogia |
|---------|------|----------|
| **Structured Streaming** | Silnik przetwarzania strumieniowego w Spark, traktuje strumień jak "nieskończoną tabelę" | Tabela, do której ciągle dopisują się nowe wiersze |
| **Source** (źródło) | Skąd przychodzą dane: Kafka, plik, socket, rate | Kran z wodą 🚰 |
| **Sink** (ujście) | Gdzie trafiają wyniki: console, memory, file, Kafka | Wiadro na wodę 🪣 |
| **Trigger** | Jak często przetwarzamy nowe dane | Częstotliwość otwierania kranu |
| **Output Mode** | Co zapisujemy: nowe wiersze (Append), zaktualizowane (Update), wszystko (Complete) | Co dokładnie wkładamy do wiadra |
| **Watermark** | Czas oczekiwania na spóźnione dane | Ile czekamy na spóźnialskich na wykładzie 😅 |
| **Window** | Okno czasowe do grupowania zdarzeń | Blok czasu: "zdarzenia z ostatnich 5 minut" |
| **Checkpoint** | Punkt kontrolny — zapisany stan strumienia | Save w grze komputerowej 🎮 |

### 🏗️ Architektura Structured Streaming

```
  Source (Kafka, files, rate)     Przetwarzanie         Sink (console, memory, files)
  ┌─────────────────────┐    ┌──────────────────┐    ┌─────────────────────────┐
  │ Dane wejściowe      │───▶│ filter / groupBy  │───▶│ Wyniki                  │
  │ (strumień)          │    │ window / watermark│    │ (ciągły zapis)          │
  └─────────────────────┘    └──────────────────┘    └─────────────────────────┘
        readStream              Transformacje              writeStream
```

### 📡 Źródła danych w Colab

W środowisku Colab nie mamy Kafki ani otwartych socketów, więc używamy:

| Źródło | Opis | Użycie |
|--------|------|--------|
| **rate** | Generuje wiersze z `timestamp` i rosnącym `value` | `spark.readStream.format('rate')` |
| **file** | Czyta nowe pliki z katalogu | `spark.readStream.format('json').load(path)` |
| **memory** (sink) | Zapisuje wyniki do tabeli w pamięci | `.format('memory').queryName('nazwa')` |
| **console** (sink) | Wypisuje wyniki na konsolę | `.format('console')` |

### 📱 Kontekst biznesowy

Jesteś inżynierem danych w **SmartCity Warszawa** 🏙️ — projekcie inteligentnego miasta. Czujniki IoT rozmieszczone po całej Warszawie wysyłają dane w czasie rzeczywistym: temperatura, wilgotność, natężenie ruchu, jakość powietrza. Twój system musi przetwarzać te dane na bieżąco!

Gotowy/a? Zaczynamy! 💪

## 🛠️ Funkcje pomocnicze — uruchom bez modyfikacji

Poniższe funkcje będą potrzebne w dalszych ćwiczeniach do symulowania danych IoT i zarządzania strumieniami.

In [ ]:
# @title 🛠️ Funkcje pomocnicze — uruchom bez modyfikacji

SENSOR_TYPES = ['temperature', 'humidity', 'traffic', 'air_quality']
DISTRICTS = ['Mokotów', 'Śródmieście', 'Wola', 'Praga-Południe', 'Ursynów', 'Bemowo']

def generate_iot_data(num_records: int = 100) -> list[dict]:
    """Generuje symulowane dane IoT czujników SmartCity Warszawa."""
    records = []
    base_time = datetime.now()
    for i in range(num_records):
        sensor_type = random.choice(SENSOR_TYPES)
        # Realistyczne wartości zależne od typu czujnika
        value_ranges = {
            'temperature': (random.uniform(-10, 40), '°C'),
            'humidity': (random.uniform(20, 95), '%'),
            'traffic': (random.randint(0, 500), 'pojazdów/h'),
            'air_quality': (random.uniform(10, 300), 'AQI'),
        }
        value, unit = value_ranges[sensor_type]
        # Dodajemy losowe opóźnienie żeby symulować "czas rzeczywisty"
        event_time = base_time + timedelta(seconds=random.uniform(-30, 30))
        records.append({
            'sensor_id': f'S-{random.randint(1, 50):03d}',
            'sensor_type': sensor_type,
            'district': random.choice(DISTRICTS),
            'value': round(value, 2) if isinstance(value, float) else value,
            'unit': unit,
            'event_time': event_time.strftime('%Y-%m-%d %H:%M:%S'),
        })
    return records


def write_iot_batch(batch_dir: str, batch_num: int, num_records: int = 50) -> str:
    """Zapisuje partię danych IoT jako plik JSON do podanego katalogu."""
    records = generate_iot_data(num_records)
    file_path = os.path.join(batch_dir, f'batch_{batch_num:04d}.json')
    with open(file_path, 'w') as f:
        for record in records:
            f.write(json.dumps(record) + '\n')  # JSON Lines format
    return file_path


def stop_all_streams():
    """Zatrzymuje wszystkie aktywne strumienie Spark."""
    for query in spark.streams.active:
        query.stop()
    print('🛑 Wszystkie strumienie zatrzymane.')


def show_stream_result(table_name: str, n: int = 20):
    """Wyświetla wyniki z tabeli strumieniowej w pamięci."""
    try:
        result = spark.sql(f'SELECT * FROM {table_name}')
        print(f'📊 Wyniki z tabeli "{table_name}" ({result.count()} wierszy):')
        result.show(n, truncate=False)
    except Exception as e:
        print(f'⚠️ Tabela "{table_name}" nie istnieje jeszcze. Poczekaj chwilę i spróbuj ponownie.')


# Seed dla powtarzalności
random.seed(42)

print('✅ Funkcje pomocnicze załadowane!')
print(f'🏙️ Typy czujników: {SENSOR_TYPES}')
print(f'📍 Dzielnice: {DISTRICTS}')

## 🎓 Przykład prowadzony — pierwszy strumień od A do Z

Zanim przejdziemy do ćwiczeń, prześledzimy razem pełny przepływ streaming — od utworzenia strumienia po odczyt wyników.

### Cel: użyć **rate source** jako generatora danych, przetworzyć je i zapisać do pamięci

In [ ]:
# @title 🎓 Przykład prowadzony — rate source → transformacja → memory sink

# KROK 1: Tworzymy strumień danych z "rate" source
# Rate source generuje wiersze z kolumnami: timestamp (czas) i value (rosnący licznik)
rate_stream = (
    spark.readStream
    .format('rate')           # Źródło: generator syntetyczny
    .option('rowsPerSecond', 10)  # 10 wierszy na sekundę
    .load()
)

# Sprawdźmy schemat — to jest "nieskończony DataFrame"
print('📋 Schemat rate source:')
rate_stream.printSchema()
print(f'🔄 Czy to streaming DataFrame? {rate_stream.isStreaming}')

# KROK 2: Transformacja — dodajemy kolumny symulujące czujnik IoT
# Używamy modulo i when() żeby przypisać typ czujnika na podstawie value
iot_demo = (
    rate_stream
    .withColumn('sensor_type',
        when(col('value') % 4 == 0, lit('temperature'))
        .when(col('value') % 4 == 1, lit('humidity'))
        .when(col('value') % 4 == 2, lit('traffic'))
        .otherwise(lit('air_quality'))
    )
    .withColumn('reading',
        _round(rand() * 100, 2)  # Losowa wartość 0-100
    )
    .select('timestamp', 'sensor_type', 'reading')
)

# KROK 3: Zapisujemy do memory sink
demo_query = (
    iot_demo.writeStream
    .format('memory')          # Sink: tabela w pamięci
    .queryName('demo_stream')  # Nazwa tabeli do SQL
    .outputMode('append')      # Tryb: dopisuj nowe wiersze
    .start()
)

print(f'\n▶️ Strumień uruchomiony! ID: {demo_query.id}')
print(f'📊 Status: {demo_query.status}')

# KROK 4: Czekamy chwilę na dane i wyświetlamy wyniki
print('\n⏳ Czekam 10 sekund na dane...')
time.sleep(10)

# Odpytujemy tabelę w pamięci jak normalną tabelę SQL!
spark.sql('SELECT * FROM demo_stream ORDER BY timestamp DESC LIMIT 10').show(truncate=False)
spark.sql('SELECT sensor_type, COUNT(*) as cnt FROM demo_stream GROUP BY sensor_type').show()

# KROK 5: ZAWSZE zatrzymuj strumień po skończeniu!
demo_query.stop()
print('🛑 Strumień demo zatrzymany.')

### 🔍 Co się właśnie wydarzyło?

1. **`readStream`** — utworzył DataFrame strumieniowy ("nieskończoną tabelę")
2. **Transformacje** (`withColumn`, `select`) — działają identycznie jak na zwykłym DataFrame!
3. **`writeStream`** — uruchomił ciągły zapis wyników do pamięci
4. **`spark.sql()`** — pozwala odpytywać dane strumieniowe jak zwykłą tabelę SQL
5. **`query.stop()`** — zatrzymuje strumień (nie zapomnij o tym! 🛑)

💡 **Kluczowy insight:** Structured Streaming to po prostu **ten sam Spark DataFrame API**, ale z `readStream` zamiast `read` i `writeStream` zamiast `write`. Jeśli znasz Spark SQL — znasz już 80% streaming!

---
Czas na samodzielne ćwiczenia! 💪

---

## ✏️ Ćwiczenie 1: Pierwszy strumień danych

**Trudność:** ⭐

Stwórz swój pierwszy strumień danych korzystając z **rate source**.

### Zadanie:
1. Utwórz streaming DataFrame z `rate` source generujący **5 wierszy na sekundę**
2. Dodaj kolumnę `event_id` — wartość `value` rzutowana na string z prefixem `"EVT-"`
3. Dodaj kolumnę `is_even` — `True` jeśli `value` jest parzysta, `False` w przeciwnym razie
4. Wybierz kolumny: `timestamp`, `event_id`, `value`, `is_even`
5. Zapisz wynik do memory sink z `queryName='cw1_events'`

**Oczekiwane kolumny:** `timestamp`, `event_id`, `value`, `is_even`

In [ ]:
# ✏️ Ćwiczenie 1: Pierwszy strumień danych

# TODO: Utwórz streaming DataFrame z rate source (5 wierszy/sekundę)
stream_cw1 = (
    spark.readStream
    .format(...)  # Zastąp ... odpowiednim formatem
    .option('rowsPerSecond', ...)  # Zastąp ... liczbą wierszy/s
    .load()
)

# TODO: Dodaj kolumny event_id i is_even, wybierz odpowiednie kolumny
events_cw1 = (
    stream_cw1
    .withColumn('event_id', ...)  # Zastąp: F.concat(lit('EVT-'), col('value').cast('string'))
    .withColumn('is_even', ...)   # Zastąp: col('value') % 2 == 0
    .select(...)                  # Zastąp: wybierz 4 kolumny
)

# TODO: Zapisz do memory sink z queryName='cw1_events' w trybie append
query_cw1 = (
    events_cw1.writeStream
    .format(...)       # Zastąp: 'memory'
    .queryName(...)    # Zastąp: 'cw1_events'
    .outputMode(...)   # Zastąp: 'append'
    .start()
)

print(f'▶️ Strumień CW1 uruchomiony! ID: {query_cw1.id}')

# Czekamy na dane
print('⏳ Czekam 15 sekund na dane...')
time.sleep(15)

# Wyświetlamy wyniki
show_stream_result('cw1_events')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 1
print("""
💡 Podpowiedź 1: Format rate tworzy kolumny 'timestamp' i 'value'
   spark.readStream.format('rate').option('rowsPerSecond', 5).load()

💡 Podpowiedź 2: Aby utworzyć event_id użyj:
   F.concat(lit('EVT-'), col('value').cast('string'))

💡 Podpowiedź 3: Aby sprawdzić parzystość:
   (col('value') % 2 == 0)

💡 Podpowiedź 4: Memory sink + append mode:
   .writeStream.format('memory').queryName('cw1_events').outputMode('append').start()
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 1

try:
    result_cw1 = spark.sql('SELECT * FROM cw1_events')
    columns = set(result_cw1.columns)

    assert result_cw1.count() > 0, '❌ Tabela cw1_events jest pusta — czy strumień działa?'
    assert 'timestamp' in columns, '❌ Brak kolumny timestamp'
    assert 'event_id' in columns, '❌ Brak kolumny event_id'
    assert 'value' in columns, '❌ Brak kolumny value'
    assert 'is_even' in columns, '❌ Brak kolumny is_even'

    # Sprawdź format event_id
    sample = result_cw1.select('event_id').first()[0]
    assert sample.startswith('EVT-'), f'❌ event_id powinien zaczynać się od "EVT-", dostałem: {sample}'

    print(f'✅ Ćwiczenie 1 zaliczone! Masz {result_cw1.count()} zdarzeń w strumieniu! 🎉')

except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Sprawdź czy strumień jest uruchomiony i poczekaj kilka sekund.')
finally:
    # Zatrzymaj strumień
    if 'query_cw1' in dir() and query_cw1.isActive:
        query_cw1.stop()
        print('🛑 Strumień CW1 zatrzymany.')

---

## ✏️ Ćwiczenie 2: Filtrowanie strumienia — czujniki temperatury

**Trudność:** ⭐⭐

Czas przefiltrować strumień danych! W SmartCity Warszawa interesują nas szczególnie **czujniki temperatury**, które przekraczają próg alarmowy.

### Zadanie:
1. Utwórz strumień z `rate` source (10 wierszy/sekundę)
2. Dodaj kolumny symulujące dane IoT:
   - `sensor_type` — na podstawie `value % 4`: 0→temperature, 1→humidity, 2→traffic, 3→air_quality
   - `reading` — losowa wartość: `F.round(F.rand() * 50, 1)` (0-50)
   - `district` — na podstawie `value % 6` — przypisz jedną z 6 dzielnic
3. **Przefiltruj** — zostaw TYLKO wiersze gdzie `sensor_type == 'temperature'` AND `reading > 30.0`
4. Zapisz do memory sink `'cw2_hot_sensors'` w trybie `'append'`

**Oczekiwany wynik:** Tylko alarmy temperaturowe z wartościami > 30°C

In [ ]:
# ✏️ Ćwiczenie 2: Filtrowanie strumienia

# TODO: Utwórz strumień rate source (10 wierszy/sekundę)
stream_cw2 = (
    spark.readStream
    .format('rate')
    .option('rowsPerSecond', 10)
    .load()
)

# TODO: Dodaj kolumny sensor_type, reading, district
iot_cw2 = (
    stream_cw2
    .withColumn('sensor_type',
        when(col('value') % 4 == 0, lit('temperature'))
        .when(col('value') % 4 == 1, lit('humidity'))
        .when(col('value') % 4 == 2, lit('traffic'))
        .otherwise(lit('air_quality'))
    )
    .withColumn('reading', ...        # TODO: F.round(F.rand() * 50, 1)
    )
    .withColumn('district',
        when(col('value') % 6 == 0, lit('Mokotów'))
        .when(col('value') % 6 == 1, lit('Śródmieście'))
        .when(col('value') % 6 == 2, lit('Wola'))
        .when(col('value') % 6 == 3, lit('Praga-Południe'))
        .when(col('value') % 6 == 4, lit('Ursynów'))
        .otherwise(lit('Bemowo'))
    )
)

# TODO: Przefiltruj — tylko temperature > 30.0
hot_sensors = (
    iot_cw2
    .filter(...)  # Zastąp: (col('sensor_type') == 'temperature') & (col('reading') > 30.0)
    .select('timestamp', 'sensor_type', 'district', 'reading')
)

# TODO: Zapisz do memory sink
query_cw2 = (
    hot_sensors.writeStream
    .format(...)          # Zastąp: 'memory'
    .queryName(...)       # Zastąp: 'cw2_hot_sensors'
    .outputMode('append')
    .start()
)

print(f'▶️ Strumień CW2 uruchomiony!')
print('⏳ Czekam 15 sekund na dane...')
time.sleep(15)

show_stream_result('cw2_hot_sensors')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 2
print("""
💡 Podpowiedź 1: Losowa wartość odczytu:
   .withColumn('reading', F.round(F.rand() * 50, 1))

💡 Podpowiedź 2: Filtrowanie — dwa warunki łączone przez &:
   .filter((col('sensor_type') == 'temperature') & (col('reading') > 30.0))

💡 Podpowiedź 3: Memory sink:
   .format('memory').queryName('cw2_hot_sensors')
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 2

try:
    result_cw2 = spark.sql('SELECT * FROM cw2_hot_sensors')

    assert result_cw2.count() > 0, '❌ Brak danych — czy strumień działa?'

    # Sprawdź, czy wszystkie to temperature
    types = [row.sensor_type for row in result_cw2.select('sensor_type').distinct().collect()]
    assert types == ['temperature'], f'❌ Powinny być tylko temperature, ale mamy: {types}'

    # Sprawdź, czy wszystkie readings > 30
    min_reading = result_cw2.agg(_min('reading')).first()[0]
    assert min_reading > 30.0, f'❌ Minimalna wartość to {min_reading}, a powinna być > 30.0'

    print(f'✅ Ćwiczenie 2 zaliczone! Wykryto {result_cw2.count()} alarmów temperaturowych! 🌡️🔥')

except Exception as e:
    print(f'❌ Błąd: {e}')
    print('Upewnij się, że strumień jest uruchomiony i dane się pojawiły.')
finally:
    if 'query_cw2' in dir() and query_cw2.isActive:
        query_cw2.stop()
        print('🛑 Strumień CW2 zatrzymany.')

---

## ✏️ Ćwiczenie 3: Agregacje na strumieniu — tryby output

**Trudność:** ⭐⭐

To jedno z najważniejszych ćwiczeń! W Structured Streaming mamy **trzy tryby wyjścia** (output modes) i każdy działa inaczej:

| Output Mode | Co zapisuje | Kiedy używać |
|-------------|-------------|-------------|
| **Append** | Tylko nowe wiersze, które nie zmienią się w przyszłości | Proste transformacje bez agregacji |
| **Complete** | Cała zaktualizowana tabela wyników | Agregacje — `groupBy`, `count` |
| **Update** | Tylko zmienione wiersze | Agregacje — wydajniejsza wersja Complete |

### Zadanie:
1. Utwórz strumień z rate source (10 wierszy/s) z kolumną `sensor_type` (jak w CW2)
2. Policz zdarzenia per `sensor_type` (`groupBy` + `count`)
3. Zapisz wyniki w trybie **Complete** do memory sink `'cw3_counts'`
4. Po 15 sekundach wyświetl wyniki

> ⚠️ **Uwaga:** Agregacje (`groupBy`) wymagają trybu **Complete** lub **Update** — tryb Append nie zadziała!

In [ ]:
# ✏️ Ćwiczenie 3: Agregacje na strumieniu

# Strumień z typami czujników
stream_cw3 = (
    spark.readStream
    .format('rate')
    .option('rowsPerSecond', 10)
    .load()
    .withColumn('sensor_type',
        when(col('value') % 4 == 0, lit('temperature'))
        .when(col('value') % 4 == 1, lit('humidity'))
        .when(col('value') % 4 == 2, lit('traffic'))
        .otherwise(lit('air_quality'))
    )
)

# TODO: Policz zdarzenia per sensor_type
counts_cw3 = (
    stream_cw3
    .groupBy(...)  # Zastąp: 'sensor_type'
    .count()
)

# TODO: Zapisz w trybie COMPLETE do memory sink 'cw3_counts'
query_cw3 = (
    counts_cw3.writeStream
    .format('memory')
    .queryName('cw3_counts')
    .outputMode(...)  # Zastąp: 'complete' — bo mamy agregację!
    .start()
)

print('▶️ Strumień CW3 uruchomiony!')
print('⏳ Czekam 15 sekund na dane...')
time.sleep(15)

show_stream_result('cw3_counts')

# Poczekaj jeszcze 10 sekund i wyświetl ponownie — wartości powinny rosnąć!
print('\n⏳ Czekam kolejne 10 sekund — obserwuj jak rosną liczby...')
time.sleep(10)

show_stream_result('cw3_counts')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 3
print("""
💡 Podpowiedź 1: Grupowanie:
   stream_cw3.groupBy('sensor_type').count()

💡 Podpowiedź 2: Dla agregacji (groupBy, count, sum, avg) MUSISZ użyć:
   .outputMode('complete')  — zwraca całą zaktualizowaną tabelę
   lub
   .outputMode('update')    — zwraca tylko zmienione wiersze

💡 Podpowiedź 3: Tryb 'append' NIE działa z agregacjami
   (bo wyniki ciągle się zmieniają — nie da się "dopisać" czegoś, co się zmieni)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 3

try:
    result_cw3 = spark.sql('SELECT * FROM cw3_counts ORDER BY count DESC')

    assert result_cw3.count() == 4, f'❌ Powinny być 4 typy czujników, jest {result_cw3.count()}'

    sensor_types = sorted([row.sensor_type for row in result_cw3.collect()])
    expected_types = sorted(['temperature', 'humidity', 'traffic', 'air_quality'])
    assert sensor_types == expected_types, f'❌ Oczekiwane typy: {expected_types}, mamy: {sensor_types}'

    total = result_cw3.agg(_sum('count')).first()[0]
    assert total > 0, '❌ Zliczenia powinny być > 0'

    print(f'✅ Ćwiczenie 3 zaliczone! Łącznie {total} zdarzeń w 4 kategoriach! 📊🎉')
    result_cw3.show()

except Exception as e:
    print(f'❌ Błąd: {e}')
finally:
    if 'query_cw3' in dir() and query_cw3.isActive:
        query_cw3.stop()
        print('🛑 Strumień CW3 zatrzymany.')

---

## ✏️ Ćwiczenie 4: Window operations — okna czasowe (tumbling window)

**Trudność:** ⭐⭐

Okna czasowe to serce przetwarzania strumieniowego! **Tumbling window** dzieli strumień na równe, nieprzekrywające się bloki czasu.

```
  ────────────────────────────────────────────────▶ czas
  |  Okno 1  |  Okno 2  |  Okno 3  |  Okno 4  |
  |  0-10s   |  10-20s  |  20-30s  |  30-40s  |
  |  ███     |  █████   |  ██      |  ████    |  ← zdarzenia
```

### Zadanie:
1. Utwórz strumień z rate source (10 wierszy/s) z kolumną `sensor_type`
2. Użyj **tumbling window** o rozmiarze **10 sekund** na kolumnie `timestamp`
3. Pogrupuj po `window` i `sensor_type`, policz zdarzenia w każdym oknie
4. Zapisz do memory sink `'cw4_windows'` w trybie `'complete'`

**Kluczowa funkcja:** `window(col('timestamp'), '10 seconds')`

> 💡 Funkcja `window()` tworzy kolumnę ze strukturą `{start, end}` — granicami okna czasowego.

In [ ]:
# ✏️ Ćwiczenie 4: Tumbling window

# Strumień z typami czujników
stream_cw4 = (
    spark.readStream
    .format('rate')
    .option('rowsPerSecond', 10)
    .load()
    .withColumn('sensor_type',
        when(col('value') % 4 == 0, lit('temperature'))
        .when(col('value') % 4 == 1, lit('humidity'))
        .when(col('value') % 4 == 2, lit('traffic'))
        .otherwise(lit('air_quality'))
    )
)

# TODO: Grupuj po tumbling window (10 sekund) i sensor_type, policz zdarzenia
windowed_cw4 = (
    stream_cw4
    .groupBy(
        window(..., ...),  # Zastąp: window(col('timestamp'), '10 seconds')
        ...                # Zastąp: col('sensor_type')
    )
    .count()
)

# TODO: Zapisz do memory sink 'cw4_windows' w trybie complete
query_cw4 = (
    windowed_cw4.writeStream
    .format('memory')
    .queryName(...)       # Zastąp: 'cw4_windows'
    .outputMode(...)      # Zastąp: 'complete'
    .start()
)

print('▶️ Strumień CW4 uruchomiony!')
print('⏳ Czekam 30 sekund na kilka okien czasowych...')
time.sleep(30)

# Wyświetl wyniki — posortowane po oknie i typie czujnika
print('📊 Wyniki — tumbling window 10 sekund:')
spark.sql("""
    SELECT window.start, window.end, sensor_type, count
    FROM cw4_windows
    ORDER BY window.start DESC, sensor_type
""").show(20, truncate=False)

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 4
print("""
💡 Podpowiedź 1: Tumbling window — nieprzekrywające się okna:
   window(col('timestamp'), '10 seconds')
   Pierwszy argument = kolumna czasu
   Drugi argument = rozmiar okna

💡 Podpowiedź 2: GroupBy z window:
   .groupBy(window(col('timestamp'), '10 seconds'), col('sensor_type'))

💡 Podpowiedź 3: W SQL wynik window ma strukturę {start, end}:
   SELECT window.start, window.end, sensor_type, count FROM ...
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 4

try:
    result_cw4 = spark.sql('SELECT * FROM cw4_windows')

    assert result_cw4.count() > 0, '❌ Brak danych w cw4_windows'
    assert 'window' in result_cw4.columns, '❌ Brak kolumny window — czy użyłeś window()?'
    assert 'sensor_type' in result_cw4.columns, '❌ Brak kolumny sensor_type'
    assert 'count' in result_cw4.columns, '❌ Brak kolumny count'

    # Sprawdź czy są przynajmniej 2 różne okna
    num_windows = result_cw4.select('window').distinct().count()
    assert num_windows >= 2, f'❌ Powinny być ≥2 okna, jest {num_windows} — może poczekaj dłużej?'

    print(f'✅ Ćwiczenie 4 zaliczone! {num_windows} okien czasowych × 4 typy czujników! 🕐🎉')

except Exception as e:
    print(f'❌ Błąd: {e}')
finally:
    if 'query_cw4' in dir() and query_cw4.isActive:
        query_cw4.stop()
        print('🛑 Strumień CW4 zatrzymany.')

---

## ✏️ Ćwiczenie 5: Sliding windows — nakładające się okna

**Trudność:** ⭐⭐⭐

**Sliding window** to okno, które „przesuwa się
,
,
,
,
1
,
2
,
3
,
4
,
5
10
,
,
,
1
2
,
,
,
3
,
4
,
5
4
,
,

: 
,
: {},
: [
5
10
,
,
,
4

,
4
1
,
4
2
,
,
,
,
,
10
,
,
,
,
,
,
,
,
,
,
10
    SELECT window.start, window.end, sensor_type, count
    FROM cw5_sliding
    ORDER BY window.start DESC, sensor_type
""").show(20, truncate=False)

# 🔍 Porównanie z tumbling window
print('\n🔍 Zauważ różnicę:')
print('   - Tumbling window (CW4): okna NIE nakładają się (np. 0-10, 10-20, 20-30)')
print('   - Sliding window (CW5): okna NAKŁADAJĄ się (np. 0-20, 10-30, 20-40)')
print('   - Efekt: w sliding window to samo zdarzenie może być W WIELU oknach!')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 5
print("""
💡 Podpowiedź 1: Sliding window — trzy argumenty:
   window(col('timestamp'), '20 seconds', '10 seconds')
   1) kolumna czasu
   2) rozmiar okna (window size)
   3) przesunięcie (slide interval)

💡 Podpowiedź 2: Jeśli slide < window, okna się nakładają
   Jeśli slide = window, to jest tumbling window
   Jeśli slide > window, tracisz dane (luki między oknami)!

💡 Podpowiedź 3: Reszta kodu identyczna jak w CW4 — zmienia się tylko window()
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 5

try:
    result_cw5 = spark.sql('SELECT * FROM cw5_sliding')

    assert result_cw5.count() > 0, '❌ Brak danych w cw5_sliding'

    # W sliding window okna się nakładają, więc powinno być ich więcej niż w tumbling
    num_windows = result_cw5.select('window').distinct().count()
    assert num_windows >= 2, f'❌ Za mało okien ({num_windows}) — poczekaj dłużej'

    # Sprawdź że okna mają po 20 sekund (nie 10)
    sample_window = result_cw5.select('window.start', 'window.end').first()
    duration = (sample_window.end - sample_window.start).total_seconds()
    assert duration == 20, f'❌ Okno powinno mieć 20s, ma {duration}s — sprawdź window()'

    print(f'✅ Ćwiczenie 5 zaliczone! Sliding window z {num_windows} nakładającymi się oknami! 🪟🎉')

except Exception as e:
    print(f'❌ Błąd: {e}')
finally:
    if 'query_cw5' in dir() and query_cw5.isActive:
        query_cw5.stop()
        print('🛑 Strumień CW5 zatrzymany.')

---

## ✏️ Ćwiczenie 6: Watermarks — obsługa spóźnionych danych

**Trudność:** ⭐⭐⭐

W rzeczywistości dane nie zawsze przychodzą na czas. Czujnik może wysłać odczyt z **opóźnieniem** — np. z powodu problemów z siecią. Watermark mówi Sparkowi: *"czekaj na spóźnione dane przez X czasu, potem zamknij okno"*.

```
  Czas zdarzenia:  10:00:05   10:00:12   10:00:08 (spóźniony!)
  Czas przybycia:  10:00:05   10:00:12   10:00:15
                                           ^
                              Ten odczyt dotarł 7 sekund po swoim event time!
                              Watermark 10s = zaakceptowany ✅
                              Watermark 5s = odrzucony ❌
```

### Zadanie:
1. Utwórz strumień z rate source
2. Dodaj **watermark** na kolumnie `timestamp` z opóźnieniem **15 sekund**
3. Użyj tumbling window **10 sekund** + grupuj po `sensor_type`
4. Policz zdarzenia w każdym oknie
5. Zapisz do memory sink `'cw6_watermark'` w trybie **`'update'`**

**Kluczowa metoda:** `.withWatermark('timestamp', '15 seconds')`

> 💡 Watermark MUSI być ustawiony PRZED grupowaniem — inaczej Spark go zignoruje!

In [ ]:
# ✏️ Ćwiczenie 6: Watermarks

# Strumień z czujnikami
stream_cw6 = (
    spark.readStream
    .format('rate')
    .option('rowsPerSecond', 10)
    .load()
    .withColumn('sensor_type',
        when(col('value') % 4 == 0, lit('temperature'))
        .when(col('value') % 4 == 1, lit('humidity'))
        .when(col('value') % 4 == 2, lit('traffic'))
        .otherwise(lit('air_quality'))
    )
)

# TODO: Dodaj watermark (15 sekund) na kolumnie timestamp
# WAŻNE: watermark PRZED groupBy!
watermarked_cw6 = (
    stream_cw6
    .withWatermark(...)  # Zastąp: 'timestamp', '15 seconds'
)

# TODO: Grupuj po window (10 sekund tumbling) i sensor_type, policz zdarzenia
windowed_cw6 = (
    watermarked_cw6
    .groupBy(
        window(..., ...),  # Zastąp: col('timestamp'), '10 seconds'
        col('sensor_type')
    )
    .count()
)

# TODO: Zapisz do memory sink 'cw6_watermark' w trybie 'update'
query_cw6 = (
    windowed_cw6.writeStream
    .format('memory')
    .queryName(...)      # Zastąp: 'cw6_watermark'
    .outputMode(...)     # Zastąp: 'update'
    .start()
)

print('▶️ Strumień CW6 z watermark uruchomiony!')
print('⏳ Czekam 30 sekund na dane...')
time.sleep(30)

# Wyświetl wyniki
print('📊 Wyniki z watermark — okna zamykane automatycznie:')
spark.sql("""
    SELECT window.start, window.end, sensor_type, count
    FROM cw6_watermark
    ORDER BY window.start, sensor_type
""").show(20, truncate=False)

print('\n💡 Watermark gwarantuje, że:')
print('   1. Dane spóźnione o ≤15s zostaną uwzględnione')
print('   2. Dane spóźnione o >15s zostaną odrzucone')
print('   3. Spark może zwolnić pamięć ze starych okien (czyszczenie stanu)')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 6
print("""
💡 Podpowiedź 1: Watermark ustawia się PRZED groupBy:
   stream.withWatermark('timestamp', '15 seconds')
   .groupBy(window(col('timestamp'), '10 seconds'), col('sensor_type'))
   .count()

💡 Podpowiedź 2: Z watermark możesz użyć outputMode('update')
   — bardziej wydajne niż 'complete', bo nie przelicza starych okien

💡 Podpowiedź 3: Watermark = max(event_time widziany dotychczas) - threshold
   Jeśli max event_time = 10:00:30 i watermark = 15s
   → akceptujemy dane z event_time ≥ 10:00:15
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 6

try:
    result_cw6 = spark.sql('SELECT * FROM cw6_watermark')

    assert result_cw6.count() > 0, '❌ Brak danych — czy strumień działa?'
    assert 'window' in result_cw6.columns, '❌ Brak kolumny window'

    num_windows = result_cw6.select('window').distinct().count()
    assert num_windows >= 1, f'❌ Powinno być ≥1 okno, jest {num_windows}'

    print(f'✅ Ćwiczenie 6 zaliczone! Watermark działa — {num_windows} okien z obsługą spóźnień! ⏰🎉')

except Exception as e:
    print(f'❌ Błąd: {e}')
finally:
    if 'query_cw6' in dir() and query_cw6.isActive:
        query_cw6.stop()
        print('🛑 Strumień CW6 zatrzymany.')

---

## ✏️ Ćwiczenie 7: File-based streaming — strumienie plikowe

**Trudność:** ⭐⭐⭐

W produkcji dane często trafiają jako **pliki** do katalogu (np. S3, GCS, HDFS). Spark Structured Streaming potrafi automatycznie wykrywać nowe pliki i przetwarzać je inkrementalnie.

```
  Katalog wejściowy:          Spark Streaming:        Wyniki:
  ┌─────────────┐            ┌──────────────┐       ┌──────────┐
  │ batch_0001 ─┼──(nowy!)──▶│ readStream   │──────▶│ memory   │
  │ batch_0002 ─┼──(nowy!)──▶│ process      │──────▶│ sink     │
  │ batch_0003 ─┼──(nowy!)──▶│ checkpoint ✓ │       │          │
  └─────────────┘            └──────────────┘       └──────────┘
```

### Zadanie:
1. Utwórz katalog na dane: `/tmp/streaming_data/iot_files`
2. Napisz kilka partii danych IoT (użyj `write_iot_batch`)
3. Utwórz strumień plikowy (`readStream.format('json')`) z podanym schematem
4. Policz zdarzenia per `sensor_type` i `district`
5. Zapisz do memory sink `'cw7_files'` w trybie `'complete'`
6. Dopisz kolejne pliki i obserwuj aktualizację wyników!

> ⚠️ **Ważne:** Przy file-based streaming MUSISZ podać schemat — Spark nie może go odczytać z jeszcze-nie-istniejących plików!

In [ ]:
# ✏️ Ćwiczenie 7: File-based streaming

# Przygotuj katalog na dane
IOT_DIR = os.path.join(DATA_DIR, 'iot_files')
IOT_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'cw7')

for d in [IOT_DIR, IOT_CHECKPOINT]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Generuj pierwszą partię danych
path_1 = write_iot_batch(IOT_DIR, batch_num=1, num_records=100)
print(f'📄 Zapisano partię 1: {path_1}')

# Schemat danych IoT — MUSISZ go podać dla file source!
iot_schema = StructType([
    StructField('sensor_id', StringType(), True),
    StructField('sensor_type', StringType(), True),
    StructField('district', StringType(), True),
    StructField('value', DoubleType(), True),
    StructField('unit', StringType(), True),
    StructField('event_time', StringType(), True),
])

# TODO: Utwórz strumień plikowy (format='json') z katalogu IOT_DIR
file_stream = (
    spark.readStream
    .format(...)       # Zastąp: 'json'
    .schema(...)       # Zastąp: iot_schema
    .load(...)         # Zastąp: IOT_DIR
)

# TODO: Grupuj po sensor_type i district, policz zdarzenia
file_counts = (
    file_stream
    .groupBy(...)  # Zastąp: 'sensor_type', 'district'
    .count()
)

# TODO: Zapisz do memory sink 'cw7_files' z checkpointem
query_cw7 = (
    file_counts.writeStream
    .format('memory')
    .queryName(...)      # Zastąp: 'cw7_files'
    .outputMode(...)     # Zastąp: 'complete'
    .option('checkpointLocation', IOT_CHECKPOINT)
    .start()
)

print('\n▶️ Strumień plikowy uruchomiony!')
print('⏳ Czekam 10 sekund na przetworzenie pierwszej partii...')
time.sleep(10)

print('\n📊 Wyniki po partii 1:')
show_stream_result('cw7_files')

In [ ]:
# Dopisz kolejne partie danych i obserwuj zmiany! (uruchom bez modyfikacji)

for batch_num in range(2, 5):
    path = write_iot_batch(IOT_DIR, batch_num=batch_num, num_records=100)
    print(f'📄 Zapisano partię {batch_num}: {path}')
    time.sleep(5)

print('\n⏳ Czekam 10 sekund na przetworzenie nowych partii...')
time.sleep(10)

print('\n📊 Wyniki po partiach 1-4:')
show_stream_result('cw7_files')

# Porównaj — liczby powinny być większe niż po partii 1!
print('\n💡 Zauważ: Spark automatycznie wykrył nowe pliki i przeliczył agregaty!')
print('   To jest potęga checkpointingu — Spark pamięta, które pliki już przetworzył.')

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 7
print("""
💡 Podpowiedź 1: File source wymaga schematu:
   spark.readStream.format('json').schema(iot_schema).load(IOT_DIR)

💡 Podpowiedź 2: GroupBy na dwóch kolumnach:
   .groupBy('sensor_type', 'district').count()

💡 Podpowiedź 3: Checkpoint jest wymagany dla file sink,
   ale też REKOMENDOWANY dla memory sink — pozwala
   na recovery po błędzie:
   .option('checkpointLocation', IOT_CHECKPOINT)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 7

try:
    result_cw7 = spark.sql('SELECT * FROM cw7_files')

    assert result_cw7.count() > 0, '❌ Brak danych w cw7_files'
    assert 'sensor_type' in result_cw7.columns, '❌ Brak kolumny sensor_type'
    assert 'district' in result_cw7.columns, '❌ Brak kolumny district'
    assert 'count' in result_cw7.columns, '❌ Brak kolumny count'

    # Powinny być dane z wielu partii
    total_events = result_cw7.agg(_sum('count')).first()[0]
    assert total_events >= 100, f'❌ Powinno być ≥100 zdarzeń, jest {total_events}'

    # Sprawdź typy czujników
    types = sorted([row.sensor_type for row in result_cw7.select('sensor_type').distinct().collect()])
    assert len(types) >= 3, f'❌ Za mało typów czujników: {types}'

    print(f'✅ Ćwiczenie 7 zaliczone! Przetworzono {total_events} zdarzeń z plików! 📂🎉')

except Exception as e:
    print(f'❌ Błąd: {e}')
finally:
    if 'query_cw7' in dir() and query_cw7.isActive:
        query_cw7.stop()
        print('🛑 Strumień CW7 zatrzymany.')

---

## 🏆 Bonus: System alertów w czasie rzeczywistym

**Trudność:** ⭐⭐⭐

Czas połączyć wszystko w pipeline produkcyjny! Zbudujesz system, który:
1. Przyjmuje strumień danych IoT
2. **Wykrywa anomalie** — wartości poza normalnym zakresem
3. Zlicza alerty w **oknach czasowych**
4. Używa **`foreachBatch`** do niestandardowego przetwarzania wyników

### Normalne zakresy wartości czujników:

| Typ czujnika | Normalny zakres | Alert gdy |
|-------------|----------------|----------|
| temperature | -10 – 35°C | > 35 lub < -10 |
| humidity | 30 – 80% | > 80 lub < 30 |
| traffic | 0 – 400 pojazdów/h | > 400 |
| air_quality | 0 – 150 AQI | > 150 |

### Zadanie:
1. Utwórz strumień z rate source i pełnymi danymi IoT (sensor_type, reading, district)
2. Dodaj kolumnę `is_anomaly` — `True` jeśli wartość poza normalnym zakresem
3. Przefiltruj — zostaw TYLKO anomalie
4. Dodaj watermark, grupuj po tumbling window (15 sekund) i `sensor_type`
5. Użyj `foreachBatch` do wypisania podsumowania alertów

> 💡 `foreachBatch` pozwala na dowolne przetwarzanie micro-batchy — możesz zapisać do bazy, wysłać maila, wywołać API!

In [ ]:
# 🏆 Bonus: System alertów

# Przechowuje historię alertów z foreachBatch
alert_history = []

# Strumień z pełnymi danymi IoT
stream_bonus = (
    spark.readStream
    .format('rate')
    .option('rowsPerSecond', 20)
    .load()
    .withColumn('sensor_type',
        when(col('value') % 4 == 0, lit('temperature'))
        .when(col('value') % 4 == 1, lit('humidity'))
        .when(col('value') % 4 == 2, lit('traffic'))
        .otherwise(lit('air_quality'))
    )
    .withColumn('district',
        when(col('value') % 6 == 0, lit('Mokotów'))
        .when(col('value') % 6 == 1, lit('Śródmieście'))
        .when(col('value') % 6 == 2, lit('Wola'))
        .when(col('value') % 6 == 3, lit('Praga-Południe'))
        .when(col('value') % 6 == 4, lit('Ursynów'))
        .otherwise(lit('Bemowo'))
    )
    .withColumn('reading',
        when(col('sensor_type') == 'temperature', _round(rand() * 60 - 15, 1))  # -15 do 45
        .when(col('sensor_type') == 'humidity', _round(rand() * 100, 1))         # 0 do 100
        .when(col('sensor_type') == 'traffic', _round(rand() * 600, 0))          # 0 do 600
        .otherwise(_round(rand() * 350, 1))                                       # 0 do 350 AQI
    )
)

# TODO: Dodaj kolumnę is_anomaly na podstawie tabeli zakresów
anomalies_bonus = (
    stream_bonus
    .withColumn('is_anomaly',
        when(
            (col('sensor_type') == 'temperature') &
            ((col('reading') > 35) | (col('reading') < -10)),
            lit(True)
        )
        .when(
            ...  # TODO: humidity — > 80 lub < 30
            lit(True)
        )
        .when(
            ...  # TODO: traffic — > 400
            lit(True)
        )
        .when(
            ...  # TODO: air_quality — > 150
            lit(True)
        )
        .otherwise(lit(False))
    )
    .filter(col('is_anomaly') == True)  # Zostaw tylko anomalie
)

# TODO: Watermark + window + grupowanie
alert_counts = (
    anomalies_bonus
    .withWatermark('timestamp', '15 seconds')
    .groupBy(
        window(col('timestamp'), '15 seconds'),
        ...  # TODO: col('sensor_type')
    )
    .agg(
        count('*').alias('alert_count'),
        _round(avg('reading'), 2).alias('avg_reading')
    )
)

# foreachBatch — niestandardowe przetwarzanie micro-batchy
def process_alerts(batch_df, batch_id):
    """Przetwarza każdy micro-batch alertów."""
    if batch_df.count() > 0:
        print(f'\n🚨 === BATCH {batch_id} — ALERTY ===  🚨')
        batch_df.select(
            col('window.start').alias('od'),
            col('window.end').alias('do'),
            'sensor_type', 'alert_count', 'avg_reading'
        ).show(truncate=False)
        # Zapisz do historii
        alert_history.append({
            'batch_id': batch_id,
            'total_alerts': batch_df.agg(_sum('alert_count')).first()[0],
        })

# Uruchom z foreachBatch
query_bonus = (
    alert_counts.writeStream
    .outputMode('update')
    .foreachBatch(process_alerts)
    .start()
)

print('▶️ System alertów uruchomiony!')
print('⏳ Oczekuję 30 sekund na alerty...')
time.sleep(30)

In [ ]:
# @title 💡 Podpowiedź — Bonus
print("""
💡 Podpowiedź 1: Warunki anomalii:
   humidity:    (col('sensor_type') == 'humidity') & ((col('reading') > 80) | (col('reading') < 30))
   traffic:     (col('sensor_type') == 'traffic') & (col('reading') > 400)
   air_quality: (col('sensor_type') == 'air_quality') & (col('reading') > 150)

💡 Podpowiedź 2: Grupowanie po window i sensor_type:
   .groupBy(window(col('timestamp'), '15 seconds'), col('sensor_type'))

💡 Podpowiedź 3: foreachBatch przyjmuje funkcję (batch_df, batch_id)
   batch_df to zwykły DataFrame — możesz na nim robić .show(), .collect(), .write itd.
""")

In [ ]:
# @title ✅ Test — Bonus

try:
    assert len(alert_history) > 0, '❌ Brak alertów — czy system działa?'

    total_all = sum(a['total_alerts'] for a in alert_history)
    print(f'✅ System alertów działa! Przetworzono {len(alert_history)} batchy, łącznie {total_all} alertów! 🚨🎉')
    print(f'\n📊 Historia batchy:')
    for a in alert_history:
        print(f"   Batch {a['batch_id']}: {a['total_alerts']} alertów")

except Exception as e:
    print(f'❌ Błąd: {e}')
finally:
    if 'query_bonus' in dir() and query_bonus.isActive:
        query_bonus.stop()
        print('🛑 System alertów zatrzymany.')

---

## 🧹 Sprzątanie — uruchom na koniec!

Zatrzymaj wszystkie strumienie i wyczyść dane tymczasowe.

In [ ]:
# @title 🧹 Sprzątanie — uruchom na koniec

# Zatrzymaj wszystkie aktywne strumienie
stop_all_streams()

# Wyczyść katalogi tymczasowe
for d in [CHECKPOINT_DIR, DATA_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f'🗑️ Usunięto: {d}')

# Zatrzymaj SparkSession
spark.stop()
print('\n🏁 Gotowe! SparkSession zamknięty.')
print('Dziękuję za udział w laboratorium! 🎉')

---

## 📝 Podsumowanie

Świetna robota! 🎉 Oto czego nauczyliśmy się dzisiaj:

| Ćwiczenie | Temat | Kluczowa umiejętność |
|-----------|-------|---------------------|
| CW1 | Pierwszy strumień | `readStream` / `writeStream` / memory sink |
| CW2 | Filtrowanie | `.filter()` na strumieniach |
| CW3 | Agregacje | Output modes: Append vs Complete vs Update |
| CW4 | Tumbling window | `window(col, '10 seconds')` |
| CW5 | Sliding window | `window(col, '20 seconds', '10 seconds')` |
| CW6 | Watermarks | `.withWatermark()` — obsługa spóźnień |
| CW7 | File streaming | JSON source + checkpointing |
| Bonus | Alert system | `foreachBatch` + anomaly detection |

### 🗝️ Najważniejsze zasady Structured Streaming:

1. **Strumień = nieskończony DataFrame** — te same operacje co w batch!
2. **Zawsze zatrzymuj strumienie** — `query.stop()` po zakończeniu pracy
3. **Output mode ma znaczenie** — Complete dla agregacji, Append dla prostych transformacji
4. **Watermark + window** — klucz do obsługi danych w czasie rzeczywistym
5. **Checkpoint** — pozwala na recovery i zapewnia exactly-once semantics

### 📚 Materiały dodatkowe:

- 📖 [Spark Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)
- 📖 [Window Operations on Event Time](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#window-operations-on-event-time)
- 📖 [Handling Late Data with Watermarks](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#handling-late-data-and-watermarking)
- 🎥 [Databricks — Deep Dive into Structured Streaming](https://www.youtube.com/watch?v=rl8dIzTpxrI)

---

> *"Dane to nowa ropa naftowa, a streaming to rurociąg, który dostarcza ją na czas."* 🛢️➡️💰

Do zobaczenia na następnych zajęciach! 👋